In [1]:
from __future__ import annotations

import json
import re
import time
from pathlib import Path
from typing import Dict

import pandas as pd

from selenium import webdriver
from selenium.common.exceptions import TimeoutException
from selenium.webdriver.common.by import By
from selenium.webdriver.firefox.options import Options as FirefoxOptions
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait

ROOT = Path.cwd()
LINKS_FILE = ROOT / "linkdownload_sezioni.txt"
DOWNLOAD_TURNOUT_DIR = ROOT / "downloads_csv"
DOWNLOAD_RESULTS_DIR = ROOT / "downloads_scrutini"
DOWNLOAD_TURNOUT_DIR.mkdir(exist_ok=True)
DOWNLOAD_RESULTS_DIR.mkdir(exist_ok=True)

DATE_TAG = "2026-03-22"


In [2]:
origin_folder = "C:\\Users\\gabri\\Documents\\GitHub\\referendum2026_mappepersezione\\scrutini_sezioni\\"

In [3]:
def parse_links_file(path: Path) -> Dict[str, str]:
    """Parsa il file testuale tipo:
    "roma":"https://...",
    "milano":"https://..."
    """
    text = path.read_text(encoding="utf-8")
    pairs = re.findall(r'"([^"]+)"\s*:\s*"([^"]+)"', text)
    if not pairs:
        raise ValueError(f"Nessun link trovato in {path}")
    return dict(pairs)


city_links = parse_links_file(LINKS_FILE)
city_links_scrutini = {city: url.replace("/votanti/", "/scrutini/") for city, url in city_links.items()}
pd.DataFrame(
    [
        {"city": city, "turnout_url": url, "results_url": city_links_scrutini[city]}
        for city, url in city_links.items()
    ]
)


,city,turnout_url,results_url
0,roma,https://elezioni.interno.gov.it/risultati/2026...,https://elezioni.interno.gov.it/risultati/2026...
1,milano,https://elezioni.interno.gov.it/risultati/2026...,https://elezioni.interno.gov.it/risultati/2026...
2,napoli,https://elezioni.interno.gov.it/risultati/2026...,https://elezioni.interno.gov.it/risultati/2026...
3,bologna,https://elezioni.interno.gov.it/risultati/2026...,https://elezioni.interno.gov.it/risultati/2026...
4,torino,https://elezioni.interno.gov.it/risultati/2026...,https://elezioni.interno.gov.it/risultati/2026...
5,genova,https://elezioni.interno.gov.it/risultati/2026...,https://elezioni.interno.gov.it/risultati/2026...
6,firenze,https://elezioni.interno.gov.it/risultati/2026...,https://elezioni.interno.gov.it/risultati/2026...
7,palermo,https://elezioni.interno.gov.it/risultati/2026...,https://elezioni.interno.gov.it/risultati/2026...
8,reggiocalabria,https://elezioni.interno.gov.it/risultati/2026...,https://elezioni.interno.gov.it/risultati/2026...
9,taranto,https://elezioni.interno.gov.it/risultati/2026...,https://elezioni.interno.gov.it/risultati/2026...


In [4]:
import re
import pandas as pd

def parse_dati(text):
    def extract(pattern, text, cast=str, transform=None, default=None):
        m = re.search(pattern, text, re.MULTILINE)
        if not m:
            return default
        value = m.group(1)
        if transform:
            value = transform(value)
        if cast:
            value = cast(value)
        return value

    dati = {
        "elettori": extract(
            r"Elettori:\s*([\d\.]+)",
            text,
            cast=int,
            transform=lambda x: x.replace(".", ""),
            default=None
        ),
        "votanti": extract(
            r"Votanti:\s*(\d+)",
            text,
            cast=int,
            default=None
        ),
        "affluenza": extract(
            r"Votanti:\s*\d+\s*\(([\d,]+)%\)",
            text,
            cast=float,
            transform=lambda x: x.replace(",", "."),
            default=None
        ),
        "schede_nulle": extract(
            r"Schede nulle:\s*(\d+)",
            text,
            cast=int,
            default=None
        ),
        "schede_bianche": extract(
            r"Schede bianche:\s*(\d+)",
            text,
            cast=int,
            default=None
        ),
        "schede_contestate": extract(
            r"Schede contestate:\s*(\d+)",
            text,
            cast=int,
            default=None
        ),
        "aggiornamento": extract(
            r"Aggiornamento:\s*(.*)",
            text,
            cast=str,
            default=None
        ),
    }

    return dati

In [5]:

def build_firefox_driver_new( headless: bool = True) -> webdriver.Firefox:
    options = FirefoxOptions()
    if headless:
        options.add_argument("-headless")

    driver = webdriver.Firefox(options=options)
    driver.set_window_size(1600, 2200)
    try:
        driver.maximize_window()
    except Exception:
        pass
    return driver


In [6]:
import tqdm

In [7]:
driver = build_firefox_driver_new(headless=False)

## scraper sezioni

In [ ]:
for city,base_scrut in city_links_scrutini.items():
    if city in ["milano","napoli"]:
        url_csv= origin_folder+f"{city}.csv"
        try:
            base_df = pd.read_csv(url_csv)
            base_df = base_df.loc[base_df["aggiornamento"].notna()]
            sezioni_fatte = base_df["sezione"].unique()
        except:
            base_df = pd.DataFrame()
            sezioni_fatte = []
        base_scrut = city_links_scrutini[city]
        ## vai all'homepage
        driver.get(base_scrut)
        time.sleep(5)
        nsezionstring = driver.find_element(By.CSS_SELECTOR, ".sezioni_perv")
        try:
            start_sezione = base_df["sezione"].max()
        except:
            start_sezione = 1
        print(f'We start at: {start_sezione}') 
        n_sezioni=int(nsezionstring.text.split("su ")[-1].replace(".",""))
        print(f'We start at: {n_sezioni}')
        risultati = []
        for sezione in range(start_sezione,n_sezioni+1):

            if sezione not in sezioni_fatte:
                sezione_url = str(sezione).zfill(4)
                sezione_url = base_scrut + sezione_url
                ## visita link sezione
                driver.get(sezione_url)
                time.sleep(1.5)
                ## prendi i risultati
                try:
                    element = driver.find_element(By.CSS_SELECTOR, ".riepilogo_basso.bg-ques-01")
                except:
                    print("aspettando")
                    time.sleep(3)
                    try:
                        element = driver.find_element(By.CSS_SELECTOR, ".riepilogo_basso.bg-ques-01")
                    except:
                        print("andando alla successiva")
                        next

                try:
                    risultato = parse_dati(element.text)
                    risultato["sezione"]=sezione
                    print(risultato)
                    risultati.append(risultato)
                    
                except:
                    next
            else:
                print("sezione fatta")
                pass
            if sezione%50==0:
                print(f'{city}{sezione}')
                cc = pd.DataFrame(risultati)
                base_df = pd.concat([base_df,cc])
                base_df.drop_duplicates(inplace=True)
                base_df.to_csv(url_csv,index=None)
    else:
        pass

We start at: 1
We start at: 445
{'elettori': 392, 'votanti': 314, 'affluenza': 80.1, 'schede_nulle': 2, 'schede_bianche': 0, 'schede_contestate': 0, 'aggiornamento': '23/03/2026 - 16:02', 'sezione': 1}
{'elettori': 581, 'votanti': 413, 'affluenza': 71.08, 'schede_nulle': 0, 'schede_bianche': 2, 'schede_contestate': 0, 'aggiornamento': '23/03/2026 - 15:55', 'sezione': 2}
{'elettori': 640, 'votanti': 459, 'affluenza': 71.72, 'schede_nulle': 0, 'schede_bianche': 1, 'schede_contestate': 0, 'aggiornamento': '23/03/2026 - 16:12', 'sezione': 3}
{'elettori': 634, 'votanti': 496, 'affluenza': 78.23, 'schede_nulle': 2, 'schede_bianche': 0, 'schede_contestate': 0, 'aggiornamento': '23/03/2026 - 16:12', 'sezione': 4}
{'elettori': 542, 'votanti': 434, 'affluenza': 80.07, 'schede_nulle': 0, 'schede_bianche': 1, 'schede_contestate': 0, 'aggiornamento': '23/03/2026 - 16:54', 'sezione': 5}
{'elettori': 638, 'votanti': 487, 'affluenza': 76.33, 'schede_nulle': 0, 'schede_bianche': 0, 'schede_contestate':

StaleElementReferenceException: Message: The element with the reference 65e6b654-41e2-482c-9ea8-665386ce9af2 is stale; either its node document is not the active document, or it is no longer connected to the DOM; For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#stale-element-reference-exception
Stacktrace:
RemoteError@chrome://remote/content/shared/RemoteError.sys.mjs:8:8
WebDriverError@chrome://remote/content/shared/webdriver/Errors.sys.mjs:202:5
StaleElementReferenceError@chrome://remote/content/shared/webdriver/Errors.sys.mjs:831:5
getKnownElement@chrome://remote/content/marionette/json.sys.mjs:412:11
deserializeJSON@chrome://remote/content/marionette/json.sys.mjs:270:20
cloneObject@chrome://remote/content/marionette/json.sys.mjs:59:24
deserializeJSON@chrome://remote/content/marionette/json.sys.mjs:298:16
json.deserialize@chrome://remote/content/marionette/json.sys.mjs:302:10
receiveMessage@chrome://remote/content

In [ ]:
print("finish")